In [6]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community langchain-text-splitters langchain-google-genai faiss-cpu

Using Python 3.11.8 environment at: C:\Users\kothu\OneDrive\Desktop\learnings\LLMOps\.venv
Audited 8 packages in 28ms


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [6]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("C:/Users/kothu/OneDrive/Desktop/learnings/LLMOps/data/Agentic_AI.txt", encoding='utf8')
document = loader.load()

In [7]:
print(document)

[Document(metadata={'source': 'C:/Users/kothu/OneDrive/Desktop/learnings/LLMOps/data/Agentic_AI.txt'}, page_content='Agentic AI is a type of artificial intelligence designed to act autonomously toward goals, rather than just respond to prompts. Think of it as AI that doesn’t just answer questions—it can plan, decide, take actions, adapt, and pursue objectives over time.\n\n🧠 Big Picture: What “Agentic” Means\n\nThe term comes from the idea of an “agent” in AI—an entity that:\n\nPerceives its environment\nMakes decisions\nTakes actions\nLearns from results\n\nTraditional AI (like chatbots) is mostly reactive.\nAgentic AI is proactive and goal-driven.\n\n⚙️ Core Characteristics of Agentic AI\n1. Goal-Oriented Behavior\n\nYou give it a high-level objective, like:\n\n“Plan a 5-day trip to Italy under $2,000”\n\nAn agentic AI can:\n\nBreak that into sub-tasks (flights, hotels, itinerary)\nExecute them step-by-step\nAdjust if constraints change\n2. Autonomy\n\nIt doesn’t need constant human 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 20)
text_chunks = text_splitter.split_documents(document)

In [9]:
text_chunks[0]

Document(metadata={'source': 'C:/Users/kothu/OneDrive/Desktop/learnings/LLMOps/data/Agentic_AI.txt'}, page_content='Agentic AI is a type of artificial intelligence designed to act autonomously toward goals, rather than just respond to prompts. Think of it as AI that doesn’t just answer questions—it can plan,')

In [10]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [11]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [12]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [13]:
query = "What are the types of AI Agents"
docs = vectorstore.similarity_search(query, k = 4)

for i, doc in enumerate(docs):
    print(f"Document {i+1}")
    print(doc.page_content)
    print("-" * 50)

Document 1
🧩 Types of Agentic AI
1. Simple Task Agents
Perform defined workflows
Example: scheduling meetings
2. Multi-Agent Systems
Multiple AIs collaborate or compete
Each has specialized roles
--------------------------------------------------
Document 2
Agentic AI is a type of artificial intelligence designed to act autonomously toward goals, rather than just respond to prompts. Think of it as AI that doesn’t just answer questions—it can plan,
--------------------------------------------------
Document 3
Personal assistants that manage your calendar, finances, and travel
Software development agents that write, test, and deploy code
Research agents that gather and synthesize information continuously
--------------------------------------------------
Document 4
3. Autonomous Agents
Operate with minimal human oversight
Long-running, adaptive systems
🚀 Examples of Agentic AI (Conceptual)
AI that runs a business process end-to-end
--------------------------------------------------


In [14]:
from langchain_core.prompts import ChatPromptTemplate
template = """You are helpful assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you dont know the answer, just say that you don't know 
Use ten sentences maximum and keep the answet short and straight
Question: {question}
Context: {context}
Answer:
"""

In [15]:
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are helpful assistant for question-answer tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you dont know the answer, just say that you don't know \nUse ten sentences maximum and keep the answet short and straight\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
output_parser = StrOutputParser()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7, max_tokens=2048, max_retries=0)
chain = prompt | llm | output_parser
context_str = "\n".join([doc.page_content for doc in docs])
response = chain.invoke({"question": query, "context": context_str})
print(response)

Based on the provided context, there are three types of AI Agents.

1.  **Simple Task Agents** perform defined workflows, such as scheduling meetings.
2.  **Multi-Agent Systems** involve multiple AIs collaborating or competing, with each having specialized roles.
3.  **Autonomous Agents** operate with minimal human oversight as long-running, adaptive systems.

Agentic AI itself is designed to act autonomously toward goals, planning rather than just responding to prompts. Examples include personal assistants, software development agents, and research agents.
